In [ ]:
import numpy as np
import MDAnalysis as mda
import MDAnalysis.transformations
from graph_utils import live_plot, live_plot_multi
from ring import vector_ring_buffer

In [ ]:
dt = 0.008  #timestep in ps
#Initialize ring buffer
window_size = 250  #size of ring buffer
buffer = vector_ring_buffer(window_size)  # ring buffer for 250 x 3D vectors
ready = False  #Flag to track if ring buffer is full
#Correlation time axis
tau_ps = np.arange(0, window_size) * dt
#Cumulative sum of MSD values for averaging
msd_sum = np.zeros(window_size)
n_samples = 0  #Number of MSD samples collected

In [ ]:
u = mda.Universe("../md/mda.tpr", "imd://localhost:8888", transformations=[mda.transformations.NoJump()])
sel = u.select_atoms("resid 1")

plot = live_plot(
    title="Mean Squared Displacement (MSD)", 
    xaxLabel="delay time (ps)", 
    yaxLabel=r"MSD (Å$^2$)",
    dataLabel = "single molecule")
# Have fixed axis for better visualization
plot["ax"].set_xlim(0, 2.0)
plot["ax"].set_ylim(0.0,10.0)

tStep = 0
#Main analysis loop
for ts in u.trajectory:
    com_position = sel.center_of_mass()
    buffer.append(com_position)
    if buffer.size == window_size:  #buffer is filled
        ready = True
        
    #Calculate MSD when window is ready
    if ready:
        history = buffer.get_values()
        ref = history[0]  #Reference position at start of window
        #Calculate MSD
        displacements = history - ref  #All past positions at once
        msd_values = np.sum(displacements**2, axis=1)  #Sum across x,y,z and reverse the order of array so that MSD of lowest tau value is first
        #Add to cumulative sum
        msd_sum += msd_values
        n_samples += 1
        #Calculate time-averaged MSD
        msd_avg = msd_sum / n_samples
    tStep += 1
    if tStep % window_size ==0:
        plot['update'](tau_ps, msd_avg)